# CHB-MIT EEG Seizure Detection - Colab Training & MLOps Pipeline

This notebook enables **free GPU-accelerated training** of the 2D-CNN Seizure Detection model. It mounts your Google Drive (where your preprocessed dataset lives), copies it to Colab's fast local SSD, configures DAGsHub remote MLflow tracking, and runs hyperparameter tuning or training at scale.

## Step 1: Mount Google Drive
Run the cell below to connect your GDrive containing the 1.6GB preprocessed `train_database_2d.h5` file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone Repository & Install Dependencies
Clone your code repository to Colab and install the MLOps and deep learning requirements.

In [ ]:
# Replace with your GitHub repository URL if private or custom
!git clone https://github.com/NeuroRoy26/seizure-detection-real-time.git
%cd seizure-detection-real-time

# Install package requirements
!pip install -q -r requirements.txt dagshub mlflow tf2onnx mne h5py pyyaml pandas scikit-learn

## Step 3: Copy Dataset to Local SSD
Reading data directly from mounted Google Drive in batches is extremely slow due to cloud network latency. Instead, we copy the HDF5 Feature Store to Colab's local high-speed SSD disk `/content`.

In [ ]:
import os
import shutil

# TODO: Update the path below to match where train_database_2d.h5 is saved in your Drive
gdrive_h5_path = "/content/drive/MyDrive/train_database_2d.h5"
local_h5_path = "/content/train_database_2d.h5"

if os.path.exists(gdrive_h5_path):
    print("[*] Copying Feature Store from Google Drive to local SSD (this takes ~30 seconds)...")
    shutil.copy(gdrive_h5_path, local_h5_path)
    print("✅ Dataset successfully copied to local SSD!")
else:
    print(f"[X] Dataset not found at: {gdrive_h5_path}")
    print("    Please verify the path to your train_database_2d.h5 file in Google Drive.")

## Step 4: Configure Local Paths & Initialize MLOps Tracking
We update the configuration file `config.yaml` to point to the local SSD path for the dataset, check if a GPU is active, and initialize DAGsHub tracking.

In [ ]:
import yaml
import tensorflow as tf
import dagshub

# 1. Point config.yaml to the local SSD dataset path
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

config["paths"]["hdf5_database_path_2d"] = "/content/train_database_2d.h5"
config["paths"]["target_onnx_path"] = "/content/latest.onnx"
config["mlflow"]["use_dagshub"] = True

with open("config.yaml", "w") as f:
    yaml.safe_dump(config, f)
print("✅ config.yaml updated for local SSD paths.")

# 2. Verify Colab GPU accelerator is active
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU ACCELERATOR DETECTED: {gpus[0]}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("⚠️ GPU NOT DETECTED. Go to Runtime -> Change runtime type -> Select 'T4 GPU'.")

# 3. Authenticate DAGsHub
# Note: When you run this cell, Colab will display a prompt or link to log in to DAGsHub.
repo_owner = config["mlflow"]["repo_owner"]
repo_name = config["mlflow"]["repo_name"]
print(f"[*] Connecting to DAGsHub: {repo_owner}/{repo_name}")
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)

## Step 5: Execute Hyperparameter Tuning Grid Search
Run the automated grid search tuner. With GPU acceleration, each 3-epoch run will complete in seconds!

In [ ]:
!python src/tune.py

## Step 6: Train the Final Production Model
Once you have analyzed the tuning results on your DAGsHub Experiments dashboard, update your final hyperparameters in `config.yaml` and run the full 20-epoch training loop.

In [ ]:
# Run final 20-epoch training
!python src/train.py

## Step 7: Save the Trained ONNX Model Weights to Google Drive
Once training completes successfully, save the resulting `latest.onnx` model weights file back to your Google Drive to ensure it is permanently stored.

In [ ]:
import shutil

onnx_dest_path = "/content/drive/MyDrive/latest.onnx"
if os.path.exists("/content/latest.onnx"):
    shutil.copy("/content/latest.onnx", onnx_dest_path)
    print(f"✅ Model saved to Google Drive: {onnx_dest_path}")
else:
    print("[X] compiled latest.onnx model file not found.")